# Decision Boundaries and Feature Crossings Demonstration

This notebook demonstrates the difference between linear and non-linear decision boundaries in Logistic Regression using a simulated circular booking geofence scenario. It illustrates how engineering polynomial features allows a linear classifier to solve complex non-linear classification tasks.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import PolynomialFeatures
from sklearn.pipeline import make_pipeline

# Set seed for reproducibility
np.random.seed(42)

# Generate 300 coordinates (x1, x2) representing users opening the app
x1 = np.random.uniform(-3, 3, 300)
x2 = np.random.uniform(-3, 3, 300)

# True label: booking (1) if within 2.0 miles of the center, else (0)
y = np.where(x1**2 + x2**2 <= 4.0, 1, 0)
df = pd.DataFrame({'x1': x1, 'x2': x2, 'y': y})

print(f"Dataset size: {len(df)} samples")
print(f"Class distribution:\n{df['y'].value_value_counts() if hasattr(df['y'], 'value_value_counts') else df['y'].value_counts()}")

## 1. Fit Linear Logistic Regression (Baseline)

First, we try to fit a standard linear logistic regression model: $z = w_1 x_1 + w_2 x_2 + b$. Since the true boundary is circular ($x_1^2 + x_2^2 \le 4$), a straight line boundary will fail to separate the classes.

In [ ]:
model_linear = LogisticRegression()
model_linear.fit(df[['x1', 'x2']], df['y'])
linear_acc = model_linear.score(df[['x1', 'x2']], df['y'])
print(f"Linear Model Accuracy: {linear_acc:.4f}")

## 2. Fit Polynomial Feature Pipeline

Next, we use `PolynomialFeatures(degree=2)` to expand our input to $x_1, x_2, x_1^2, x_2^2, x_1 x_2$. This maps the linear solver into a quadratic space, allowing it to define a circular boundary: $w_1 x_1 + w_2 x_2 + w_3 x_1^2 + w_4 x_2^2 + w_5 x_1 x_2 + b = 0$.

In [ ]:
model_poly = make_pipeline(
    PolynomialFeatures(degree=2, include_bias=False),
    LogisticRegression()
)
model_poly.fit(df[['x1', 'x2']], df['y'])
poly_acc = model_poly.score(df[['x1', 'x2']], df['y'])
print(f"Polynomial Model Accuracy: {poly_acc:.4f}")

## 3. Visualize Decision Boundaries

Let's plot both models' predictions and their respective decision boundaries in 2D feature space.

In [ ]:
# Create grid for decision boundary visualization
xx, yy = np.meshgrid(np.linspace(-3, 3, 500), np.linspace(-3, 3, 500))
grid_pts = np.c_[xx.ravel(), yy.ravel()]

# Predictions on grid points
Z_linear = model_linear.predict(grid_pts).reshape(xx.shape)
Z_poly = model_poly.predict(grid_pts).reshape(xx.shape)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Plot Linear Model Results
axes[0].contourf(xx, yy, Z_linear, alpha=0.2, colors=['#adb5bd', '#339af0'])
axes[0].scatter(df['x1'][df['y'] == 1], df['x2'][df['y'] == 1], color='#339af0', alpha=0.6, label='Booking (1)', s=15)
axes[0].scatter(df['x1'][df['y'] == 0], df['x2'][df['y'] == 0], color='#868e96', alpha=0.6, label='No Booking (0)', s=15)
# Plot decision boundary line: w1 x1 + w2 x2 + b = 0
w = model_linear.coef_[0]
b = model_linear.intercept_[0]
x1_boundary = np.linspace(-3, 3, 100)
x2_boundary = -(w[0] * x1_boundary + b) / w[1]
axes[0].plot(x1_boundary, x2_boundary, color='#e03131', linewidth=2, label='Linear boundary')
axes[0].set_xlim(-3, 3)
axes[0].set_ylim(-3, 3)
axes[0].set_title(f"Linear Logistic Regression (Acc: {linear_acc:.2%})")
axes[0].legend(loc='upper right')
axes[0].grid(True, linestyle='--')

# Plot Polynomial Model Results
axes[1].contourf(xx, yy, Z_poly, alpha=0.2, colors=['#adb5bd', '#339af0'])
axes[1].scatter(df['x1'][df['y'] == 1], df['x2'][df['y'] == 1], color='#339af0', alpha=0.6, label='Booking (1)', s=15)
axes[1].scatter(df['x1'][df['y'] == 0], df['x2'][df['y'] == 0], color='#868e96', alpha=0.6, label='No Booking (0)', s=15)
# Plot circular boundary (true boundary circle)
theta = np.linspace(0, 2*np.pi, 200)
axes[1].plot(2 * np.cos(theta), 2 * np.sin(theta), color='#2b8a3e', linewidth=2, label='True Circle boundary')
axes[1].set_xlim(-3, 3)
axes[1].set_ylim(-3, 3)
axes[1].set_title(f"Polynomial Logistic Regression (Acc: {poly_acc:.2%})")
axes[1].legend(loc='upper right')
axes[1].grid(True, linestyle='--')

plt.tight_layout()
plt.show()